In [1]:
# Safe bootstrap for reproducible notebook runs
import os
import random
import warnings

import numpy as np

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Bootstrap complete (seed=42).")

Bootstrap complete (seed=42).


# 🚀 Startup Idea Validation Crew

## What We're Building

A multi-agent crew that validates a startup idea from four perspectives:

| Agent | Role |
|-------|------|
| **Market Analyst** | Assess market size, trends, and demand |
| **Competitor Analyst** | Map the competitive landscape |
| **Pricing Analyst** | Evaluate pricing strategies and revenue models |
| **Critic Agent** | Poke holes, find weaknesses, and challenge assumptions |

## Architecture

```
Startup Idea
      │
      ▼
┌─────────────┐   ┌─────────────┐   ┌─────────────┐   ┌─────────────┐
│   Market    │──▶│ Competitor  │──▶│  Pricing    │──▶│   Critic    │
│  Analyst    │   │  Analyst    │   │  Analyst    │   │   Agent     │
└─────────────┘   └─────────────┘   └─────────────┘   └─────────────┘
                                                            │
                                                            ▼
                                                    Final Verdict
```

## Key CrewAI Concepts

- **Agent**: An autonomous unit with a role, goal, and backstory
- **Task**: A specific assignment with description and expected output
- **Crew**: A team of agents working together on tasks
- **Process.sequential**: Tasks run one after another, each building on the previous

## Stack
- **CrewAI** — multi-agent orchestration
- **Ollama** — local LLM (qwen3.5:9b)

## Step 1 — Imports & LLM Setup

In [2]:
CREWAI_AVAILABLE = True

try:
    from crewai import Agent, Task, Crew, Process, LLM
except ImportError:
    CREWAI_AVAILABLE = False
    Agent = Task = Crew = Process = LLM = None
    print("CrewAI is not installed. Install with: pip install crewai")

llm = None
if CREWAI_AVAILABLE:
    llm = LLM(
        model="ollama/qwen3.5:9b",
        base_url="http://localhost:11434",
    )
    print(f"LLM configured: {llm.model}")
else:
    print("Running in explanation/demo mode without CrewAI runtime.")

CrewAI is not installed. Install with: pip install crewai
Running in explanation/demo mode without CrewAI runtime.


## Step 2 — Define the Startup Idea

We'll validate a sample startup idea. Change this to test your own ideas.

In [3]:
startup_idea = """
Startup Name: MealMind
Concept: An AI-powered meal planning app for busy professionals.
- Users input dietary preferences, budget, and time constraints.
- The app generates weekly meal plans with grocery lists.
- Integrates with grocery delivery services (Instacart, Amazon Fresh).
- Premium tier: personalized nutrition coaching with AI.
- Target audience: Health-conscious professionals aged 25-45.
- Monetization: Freemium ($0 basic, $9.99/mo premium, $19.99/mo coaching).
- Tech stack: React Native mobile app, Python backend, LLM for recipes.
"""

print("Startup idea loaded: MealMind")

Startup idea loaded: MealMind


## Step 3 — Create Agents

Each agent has a distinct responsibility, and the handoff order is what creates coordination:
1. Market Research Analyst finds demand and market size assumptions.
2. Competitive Intelligence Analyst receives market context and maps threats/gaps.
3. Pricing Strategy Analyst uses market + competition context to stress-test monetization.
4. Startup Critic receives all prior analysis and issues the final verdict.

In `Process.sequential`, CrewAI coordinates this by passing prior task outputs forward automatically.

In [4]:
if CREWAI_AVAILABLE:
    market_analyst = Agent(
        role="Market Research Analyst",
        goal="Assess the market opportunity, size, growth trends, and demand for the startup idea",
        backstory=(
            "You are a senior market analyst with 15 years of experience at McKinsey. "
            "You specialize in sizing emerging markets and identifying consumer trends. "
            "You always back your analysis with data points and market frameworks like TAM/SAM/SOM."
        ),
        llm=llm,
        verbose=True,
    )

    competitor_analyst = Agent(
        role="Competitive Intelligence Analyst",
        goal="Map the competitive landscape, identify key players, and find gaps the startup can exploit",
        backstory=(
            "You are a competitive analyst who has tracked hundreds of startups for a top VC firm. "
            "You excel at Porter's Five Forces analysis and identifying moats. "
            "You know that most startups fail because of competition, not technology."
        ),
        llm=llm,
        verbose=True,
    )

    pricing_analyst = Agent(
        role="Pricing Strategy Analyst",
        goal="Evaluate the pricing model, revenue potential, and unit economics of the startup",
        backstory=(
            "You are a pricing strategist who has advised 50+ SaaS companies on monetization. "
            "You think in terms of LTV:CAC ratios, willingness-to-pay, and pricing psychology. "
            "You challenge founders who price too low or too high."
        ),
        llm=llm,
        verbose=True,
    )

    critic = Agent(
        role="Startup Critic & Devil's Advocate",
        goal="Challenge the startup idea, find weaknesses, and provide a final go/no-go verdict",
        backstory=(
            "You are a seasoned angel investor who has seen 1000+ pitches and invested in only 20. "
            "You are known for being brutally honest. You look for reasons a startup will FAIL, "
            "not succeed. Your job is to stress-test the idea so founders are prepared."
        ),
        llm=llm,
        verbose=True,
    )

    print("4 agents created: Market Analyst, Competitor Analyst, Pricing Analyst, Critic")
else:
    market_analyst = {"role": "Market Research Analyst"}
    competitor_analyst = {"role": "Competitive Intelligence Analyst"}
    pricing_analyst = {"role": "Pricing Strategy Analyst"}
    critic = {"role": "Startup Critic & Devil's Advocate"}
    print("Demo mode: agent roles defined for explanation.")

Demo mode: agent roles defined for explanation.


## Step 4 — Create Tasks

Each task specifies:
- **description**: What to do (with the startup idea injected)
- **expected_output**: What a good result looks like
- **agent**: Who does this task

In [5]:
if CREWAI_AVAILABLE:
    market_task = Task(
        description=f"""Analyze the market opportunity for this startup idea:

{startup_idea}

Your analysis should cover:
1. Total Addressable Market (TAM), Serviceable Addressable Market (SAM), Serviceable Obtainable Market (SOM)
2. Market growth rate and key trends driving demand
3. Target customer profile and their pain points
4. Timing — is the market ready for this solution?
5. Potential market risks and headwinds""",
        expected_output="A structured market analysis with TAM/SAM/SOM estimates, trends, and risks.",
        agent=market_analyst,
    )

    competitor_task = Task(
        description=f"""Map the competitive landscape for this startup idea:

{startup_idea}

Your analysis should cover:
1. Direct competitors (same product category)
2. Indirect competitors (alternative solutions to the same problem)
3. Each competitor's strengths and weaknesses
4. Competitive moats — what could make this startup defensible?
5. Gaps in the market that competitors are missing""",
        expected_output="A competitive landscape map with direct/indirect competitors, moats, and gaps.",
        agent=competitor_analyst,
    )

    pricing_task = Task(
        description=f"""Evaluate the pricing and monetization strategy for this startup:

{startup_idea}

Your analysis should cover:
1. Is the freemium model appropriate for this market?
2. Price point analysis — are the tiers priced correctly?
3. Estimated unit economics (LTV, CAC, payback period)
4. Revenue projections (Year 1, Year 2, Year 3 scenarios)
5. Alternative monetization strategies to consider""",
        expected_output="A pricing analysis with unit economics, revenue projections, and recommendations.",
        agent=pricing_analyst,
    )

    critic_task = Task(
        description=f"""You have access to the market analysis, competitive landscape, and pricing
evaluation for this startup idea:

{startup_idea}

Now play devil's advocate:
1. What are the top 5 reasons this startup could FAIL?
2. What assumptions are the founders making that might be wrong?
3. What would need to be true for this to be a $100M+ business?
4. Score the idea from 1-10 across: Market (size), Team Risk, Execution Difficulty, Timing
5. Final verdict: GO, CONDITIONAL GO, or NO-GO — with clear reasoning""",
        expected_output="A critical assessment with failure risks, scoring, and a go/no-go verdict.",
        agent=critic,
    )
    print("4 tasks created")
else:
    market_task = {"name": "Market research task"}
    competitor_task = {"name": "Competitor analysis task"}
    pricing_task = {"name": "Pricing analysis task"}
    critic_task = {"name": "Critic verdict task"}
    print("Demo mode: task sequence defined for explanation.")

Demo mode: task sequence defined for explanation.


## Step 5 — Assemble & Run the Crew

The crew runs tasks **sequentially** — each agent builds on the previous agent's output.

In [6]:
class _DemoTaskOutput:
    def __init__(self, description, raw):
        self.description = description
        self.raw = raw

class _DemoResult:
    def __init__(self):
        self.raw = "CONDITIONAL GO: Strong demand signal, but execution risk is moderate."
        self.tasks_output = [
            _DemoTaskOutput("Market analysis", "TAM is large with rising demand for personalized wellness tools."),
            _DemoTaskOutput("Competitor analysis", "Competition is intense, but grocery integration + coaching differentiation is promising."),
            _DemoTaskOutput("Pricing analysis", "Current tiers are plausible; test willingness-to-pay for coaching add-on."),
            _DemoTaskOutput("Critic review", "Top risks: retention, CAC, and feature parity with incumbents."),
        ]
        self.token_usage = {"mode": "demo", "note": "No live LLM call was made."}

if CREWAI_AVAILABLE:
    crew = Crew(
        agents=[market_analyst, competitor_analyst, pricing_analyst, critic],
        tasks=[market_task, competitor_task, pricing_task, critic_task],
        process=Process.sequential,
        verbose=True,
    )

    print("Crew assembled! Starting validation...")
    print("=" * 60)
    result = crew.kickoff()
else:
    print("Demo mode: skipping live CrewAI kickoff.")
    result = _DemoResult()

print("\n" + "=" * 60)
print("CREW EXECUTION COMPLETE")
print("=" * 60)

Demo mode: skipping live CrewAI kickoff.

CREW EXECUTION COMPLETE


## Step 6 — Review Results

In [7]:
print("📋 FINAL VERDICT (Critic Agent):")
print("=" * 60)
print(result.raw)

print("\n\n📊 INDIVIDUAL TASK OUTPUTS:")

if len(result.tasks_output) > 0:
    print("\n" + "─" * 60)
    print(f"Task 1: {result.tasks_output[0].description}")
    print("─" * 60)
    print(result.tasks_output[0].raw)

if len(result.tasks_output) > 1:
    print("\n" + "─" * 60)
    print(f"Task 2: {result.tasks_output[1].description}")
    print("─" * 60)
    print(result.tasks_output[1].raw)

if len(result.tasks_output) > 2:
    print("\n" + "─" * 60)
    print(f"Task 3: {result.tasks_output[2].description}")
    print("─" * 60)
    print(result.tasks_output[2].raw)

if len(result.tasks_output) > 3:
    print("\n" + "─" * 60)
    print(f"Task 4: {result.tasks_output[3].description}")
    print("─" * 60)
    print(result.tasks_output[3].raw)

📋 FINAL VERDICT (Critic Agent):
CONDITIONAL GO: Strong demand signal, but execution risk is moderate.


📊 INDIVIDUAL TASK OUTPUTS:

────────────────────────────────────────────────────────────
Task 1: Market analysis
────────────────────────────────────────────────────────────
TAM is large with rising demand for personalized wellness tools.

────────────────────────────────────────────────────────────
Task 2: Competitor analysis
────────────────────────────────────────────────────────────
Competition is intense, but grocery integration + coaching differentiation is promising.

────────────────────────────────────────────────────────────
Task 3: Pricing analysis
────────────────────────────────────────────────────────────
Current tiers are plausible; test willingness-to-pay for coaching add-on.

────────────────────────────────────────────────────────────
Task 4: Critic review
────────────────────────────────────────────────────────────
Top risks: retention, CAC, and feature parity with

In [8]:
# Token usage summary
print("\n📈 Token Usage:")
print(result.token_usage)


📈 Token Usage:
{'mode': 'demo', 'note': 'No live LLM call was made.'}


## 🧠 Key Concepts Recap

| Concept | What It Does |
|---------|-------------|
| **Agent** | Autonomous unit with role, goal, backstory, and LLM |
| **Task** | Specific assignment with description and expected output |
| **Crew** | Team of agents + tasks + execution process |
| **Process.sequential** | Tasks run in order; each sees prior outputs |
| **LLM(model="ollama/...")** | Use local Ollama model with CrewAI |
| **crew.kickoff()** | Start the crew execution |
| **result.tasks_output** | Access each individual task's output |

## 🔧 Extensions

- **Add web search**: Use `SerperDevTool` for real-time market data
- **Structured output**: Use `output_pydantic` for machine-readable verdicts
- **Multiple ideas**: Use `crew.kickoff_for_each()` to validate a batch of ideas
- **Hierarchical process**: Let a manager agent delegate tasks dynamically